## Stage 3a: Vital signs extraction with DuckDB
Extract a subset of vital sign measurements from the MIMIC-IV chartevents table using DuckDB. Only the measurements needed for patient similarity search are retained, reducing the dataset size substantially before feature engineering.

### Tasks
- Import libraries
- Load df_stage2 and d_items csv files
- Search for item ids of vital signs of interest and save them in a list
- Extract a subset of vital sign measurement from the MIMIC-IV chartevents table using DuckDB and save the csv to vitals_filtered.csv
- Clean df_vitals subset to remove physiologically impossible values and groupby vital_name
- save cleaned data to vitals_filtered_clean.csv

## Import libraries

In [ ]:
import duckdb
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
notebook_dir = os.getcwd() 
project_root = os.path.dirname(notebook_dir)  
output_dir = os.path.join(project_root, 'data', 'processed') 
os.makedirs(output_dir, exist_ok=True)

In [ ]:
df_stage2 = pd.read_csv(
    os.path.join(output_dir, 'icu_stay_features_stage2.csv')
)
print(df_stage2.shape)

In [ ]:
# Load the d_items.csv file using the environment variable
d_items_data_path = os.getenv('D_ITEMS_DATA_PATH')
df_items = pd.read_csv(d_items_data_path)

In [ ]:
# Discover the itemids for vitals of interest
# df_items[
#     df_items['label']
#     .str.contains('Heart Rate', case=False, na=False)
# ]

# df_items[
#     df_items['label']
#     .str.contains('Respiratory', case=False, na=False)
# ]

# df_items[
#     df_items['label']
#     .str.contains('Temperature', case=False, na=False)
# ]

# df_items[
#     df_items['label']
#     .str.contains('SpO2|O2 Saturation', regex=True, case=False, na=False)
# ]

df_items[
    df_items['label']
    .str.contains('Blood Pressure', case=False, na=False)
]

In [ ]:
vital_itemids = {
    "heart_rate": [220045],
    "resp_rate": [220210],
    "temperature": [223761],
    "spo2": [220277],
    "sbp": [220050]
}


In [ ]:
# Create the item list for the vitals we want to extract
item_ids = []

for ids in vital_itemids.values():
    item_ids.extend(ids)

item_ids 

In [ ]:
# Connect to DuckDB
con = duckdb.connect()

In [ ]:
# Read only the columns we need from the large CSV file
chartevents_path = os.getenv('CHARTEVENTS_DATA_PATH')
query = f"""
SELECT 
    stay_id,
    itemid,
    charttime,
    valuenum
FROM read_csv_auto('{chartevents_path}')
WHERE 
    itemid IN ({",".join(map(str, item_ids))})
    AND valuenum IS NOT NULL
"""

In [ ]:
df_vitals = con.execute(query).fetch_df()

print(df_vitals.shape)
df_vitals.head()

In [ ]:
# Map itemid to a readable name
item_map = {
    220045: 'heart_rate',
    220210: 'resp_rate',      
    223761: 'temperature',
    220277: 'spo2',
    220050: 'sbp'
}
df_vitals['vital_name'] = df_vitals['itemid'].map(item_map)

In [ ]:
# Save the extracted vitals to a CSV file
output_file = os.path.join(output_dir, 'vitals_filtered_clean.csv')
df_vitals.to_csv(output_file, index=False)
print(f"{len(df_vitals):,} extracted vitals saved to {output_file}")

In [ ]:
# 
valid_ranges = {
    'heart_rate': (20, 250),
    'resp_rate': (5, 80),
    'temperature': (90, 110),
    'spo2': (50, 100),
    'sbp': (40, 300)
}

In [ ]:
# Count the outliers
removed_counts = []

for vital, (low, high) in valid_ranges.items():

    mask = df_vitals["vital_name"] == vital

    outliers = (
        mask &
        (
            (df_vitals["valuenum"] < low) |
            (df_vitals["valuenum"] > high)
        )
    )

    removed_counts.append({
        "vital": vital,
        "removed": outliers.sum(),
        "total": mask.sum(),
        "percent_removed": 100 * outliers.sum() / mask.sum()
    })

removed_df = pd.DataFrame(removed_counts)

removed_df

In [ ]:
df_vitals_clean = df_vitals.copy()

for vital, (low, high) in valid_ranges.items():
    mask = df_vitals_clean["vital_name"] == vital

    df_vitals_clean.loc[
        mask &
        (
            (df_vitals_clean["valuenum"] < low) |
            (df_vitals_clean["valuenum"] > high)
        ),
        "valuenum"
    ] = pd.NA

In [ ]:
df_vitals_clean = df_vitals_clean.dropna(subset=['valuenum'])

In [ ]:
df_vitals_clean.groupby('vital_name')['valuenum'].describe()

In [ ]:
# Save the extracted vitals to a CSV file
output_file = os.path.join(output_dir, 'vitals_filtered.csv')
df_vitals_clean.to_csv(output_file, index=False)
print(f"Saved the cleaned vitals to CSV")

In [ ]:
print(df_vitals['vital_name'].value_counts)

In [ ]:
df_vitals.shape

In [ ]:
df_vitals.head()

In [ ]:
df_vitals.isnull().sum()

In [ ]:
df_vitals['stay_id'].nunique()

In [ ]:
df_vitals.groupby('vital_name')['valuenum'].describe()

### Summary
Only the measurements needed for patient similarity search are retained, reducing the dataset size substantially before feature engineering.

Initial exploration of the extracted vital signs revealed a small number of physiologically impossible values (e.g., heart rate >10,000,000 bpm and negative SpO₂). These are consistent with known artifacts in bedside monitoring systems and clinical data entry rather than true patient measurements. To improve data quality, clinically informed thresholds were applied to each vital sign, and values outside these ranges were removed before feature aggregation. The resulting distributions aligned with expected ICU physiological ranges.

Quality Control: Outlier detection showed that less than a small fraction of measurements fell outside clinically plausible physiological ranges. These observations were removed prior to aggregation to prevent monitor artifacts and data-entry errors from influencing patient similarity features.